# Part 1: Data Preprocessing - Zomato Bangalore Dataset
## Comparative Analysis of Transformer Models for Aspect-Based Sentiment Analysis

**Dataset:** Zomato Bangalore Restaurant Reviews (~51K restaurants)  
**Source:** https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants

---

### 📋 What This Notebook Does:
1. Load Zomato Bangalore dataset
2. Extract individual reviews from `reviews_list` column
3. Parse ratings and clean text
4. Create sentiment labels from ratings
5. Extract aspects (Food, Service, Ambiance, Price)
6. Prepare data splits for training

---

## Step 1: Install & Import Libraries

In [ ]:
# Install required packages
!pip install pandas numpy matplotlib seaborn nltk scikit-learn -q

print("✓ Packages installed!")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import ast
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully!")

## Step 2: Download NLTK Data

In [ ]:
import nltk

print("Downloading NLTK data...")
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("✓ NLTK data downloaded!")

## Step 3: Load Dataset

### 📥 Before Running:
1. Download: **[Zomato Bangalore Dataset](https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants)**
2. Upload the CSV file to this notebook directory
3. Update the filename below if needed

In [ ]:
# ⚠️ UPDATE THIS with your actual filename
DATASET_PATH = "zomato.csv"

print(f"Loading dataset: {DATASET_PATH}")
print("="*70)

try:
    df = pd.read_csv(DATASET_PATH)
    print(f"✓ Successfully loaded {len(df):,} restaurants\n")
    
    print(f"Dataset shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}\n")
    
    # Display first few rows
    print("First 3 rows:")
    display(df.head(3))
    
except FileNotFoundError:
    print(f"❌ ERROR: File '{DATASET_PATH}' not found!\n")
    print("Please:")
    print("1. Download from: https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants")
    print("2. Upload CSV to this notebook")
    print("3. Update DATASET_PATH above")
    raise

## Step 4: Explore Dataset

In [ ]:
print("📊 DATASET STATISTICS")
print("="*70)

print(f"\nTotal restaurants: {len(df):,}")
print(f"Total columns: {len(df.columns)}")

print("\nMissing values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
display(missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False))

In [ ]:
# Check the reviews_list column
print("\n📝 Checking 'reviews_list' column...\n")
print(f"Non-null reviews: {df['reviews_list'].notna().sum():,} / {len(df):,}")
print(f"\nSample review entry:")
print(df['reviews_list'].iloc[0])

## Step 5: Extract Individual Reviews

The `reviews_list` column contains multiple reviews per restaurant as a string representation of a list of tuples:
```python
[("Rated 4.0", "Great food!"), ("Rated 3.5", "Good service"), ...]
```

We'll expand this into individual review rows.

In [ ]:
def extract_reviews_from_restaurant(row):
    """
    Extract individual reviews from a restaurant's reviews_list.
    Returns a list of dictionaries with review data.
    """
    reviews_data = []
    
    if pd.isna(row['reviews_list']) or str(row['reviews_list']).strip() == '[]':
        return reviews_data
    
    try:
        # Convert string representation to actual list
        reviews_list = ast.literal_eval(str(row['reviews_list']))
        
        for review_tuple in reviews_list:
            if len(review_tuple) >= 2:
                rating_str = str(review_tuple[0])
                review_text = str(review_tuple[1])
                
                # Extract numeric rating from "Rated 4.0" format
                rating_match = re.search(r'(\d+\.?\d*)', rating_str)
                
                if rating_match and review_text and review_text.lower() != 'none':
                    rating = float(rating_match.group(1))
                    
                    # Create review entry with restaurant context
                    reviews_data.append({
                        'restaurant_name': row['name'],
                        'review_text': review_text,
                        'rating': rating,
                        'location': row['location'],
                        'cuisines': row['cuisines'],
                        'rest_type': row['rest_type'],
                        'online_order': row['online_order'],
                        'book_table': row['book_table'],
                        'approx_cost': row['approx_cost(for two people)']
                    })
    except Exception as e:
        pass  # Skip malformed entries
    
    return reviews_data

print("✓ Review extraction function defined")

In [ ]:
# Extract all reviews
print("\nExtracting individual reviews from all restaurants...")
print("This may take 2-3 minutes...\n")

all_reviews = []
for idx, row in df.iterrows():
    reviews = extract_reviews_from_restaurant(row)
    all_reviews.extend(reviews)
    
    if (idx + 1) % 10000 == 0:
        print(f"Processed {idx + 1:,} / {len(df):,} restaurants... Found {len(all_reviews):,} reviews so far")

# Create new dataframe from extracted reviews
reviews_df = pd.DataFrame(all_reviews)

print(f"\n✓ Extraction complete!")
print(f"  Total reviews extracted: {len(reviews_df):,}")
print(f"  From {len(df):,} restaurants")
print(f"  Average reviews per restaurant: {len(reviews_df)/len(df):.1f}")

In [ ]:
# Display sample extracted reviews
print("\n📝 Sample Extracted Reviews:")
print("="*70)
display(reviews_df.head(10))

## Step 6: Clean Review Text

In [ ]:
def clean_text(text):
    """
    Clean review text.
    """
    if pd.isna(text):
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove special characters but keep punctuation
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'\-]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

print("Cleaning review text...")
reviews_df['cleaned_text'] = reviews_df['review_text'].apply(clean_text)

# Remove very short reviews (less than 10 characters)
reviews_df = reviews_df[reviews_df['cleaned_text'].str.len() >= 10].copy()

print(f"✓ Text cleaning completed!")
print(f"  Reviews after cleaning: {len(reviews_df):,}")

# Show examples
print("\n📝 Cleaning Examples:")
print("="*70)
for i in range(3):
    print(f"\nReview {i+1}:")
    print(f"  Original: {reviews_df['review_text'].iloc[i][:100]}...")
    print(f"  Cleaned:  {reviews_df['cleaned_text'].iloc[i][:100]}...")

## Step 7: Create Sentiment Labels

Convert ratings (typically 1-5) to sentiment:
- **Positive:** 4-5 stars
- **Neutral:** 3 stars
- **Negative:** 1-2 stars

In [ ]:
# Check rating distribution
print("Rating Distribution:")
print("="*70)
print(reviews_df['rating'].value_counts().sort_index())
print(f"\nRating range: {reviews_df['rating'].min()} - {reviews_df['rating'].max()}")

In [ ]:
def create_sentiment_label(rating):
    """
    Convert rating to sentiment label.
    """
    if pd.isna(rating):
        return 'neutral'
    
    rating = float(rating)
    
    if rating >= 4:
        return 'positive'
    elif rating <= 2:
        return 'negative'
    else:
        return 'neutral'

# Create sentiment labels
print("\nCreating sentiment labels...\n")
reviews_df['sentiment'] = reviews_df['rating'].apply(create_sentiment_label)

# Create numeric labels
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
reviews_df['label'] = reviews_df['sentiment'].map(label_map)

print("✓ Sentiment labels created!\n")

# Show distribution
print("Sentiment Distribution:")
print("="*70)
sentiment_counts = reviews_df['sentiment'].value_counts()
for sentiment in ['positive', 'neutral', 'negative']:
    count = sentiment_counts.get(sentiment, 0)
    pct = count / len(reviews_df) * 100
    print(f"  {sentiment.capitalize():10}: {count:6,} reviews ({pct:5.1f}%)")

print(f"\n  Total: {len(reviews_df):,} reviews")

In [ ]:
# Visualize sentiment distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Pie chart
colors = ['#2ca02c', '#ffbb78', '#d62728']
sentiment_counts = reviews_df['sentiment'].value_counts()
ax1.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%',
        colors=colors, startangle=90)
ax1.set_title('Sentiment Distribution', fontweight='bold', fontsize=14)

# Bar chart
sentiment_counts.plot(kind='bar', ax=ax2, color=colors)
ax2.set_title('Review Count by Sentiment', fontweight='bold', fontsize=14)
ax2.set_xlabel('Sentiment', fontweight='bold')
ax2.set_ylabel('Number of Reviews', fontweight='bold')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Extract Aspects

Identify mentions of:
- **Food:** Quality, taste, dishes
- **Service:** Staff, waiters, attention
- **Ambiance:** Atmosphere, decor
- **Price:** Value, cost

In [ ]:
# Define aspect keywords
ASPECT_KEYWORDS = {
    'food': [
        'food', 'dish', 'dishes', 'meal', 'taste', 'flavor', 'flavour',
        'delicious', 'tasty', 'yummy', 'cuisine', 'menu', 'plate',
        'recipe', 'ingredient', 'cooked', 'fresh', 'quality',
        'pizza', 'pasta', 'biryani', 'dosa', 'idli', 'curry',
        'burger', 'sandwich', 'soup', 'dessert', 'sweet'
    ],
    'service': [
        'service', 'waiter', 'waiters', 'waitress', 'staff', 'server',
        'friendly', 'attentive', 'helpful', 'professional', 'courteous',
        'rude', 'slow', 'quick', 'fast', 'efficient', 'prompt',
        'wait', 'waited', 'waiting', 'attention', 'manager', 'hospitality'
    ],
    'ambiance': [
        'ambiance', 'ambience', 'atmosphere', 'decor', 'decoration',
        'environment', 'setting', 'vibe', 'music', 'lighting',
        'comfortable', 'cozy', 'romantic', 'casual', 'elegant',
        'noisy', 'loud', 'quiet', 'crowded', 'spacious', 'interior',
        'seating', 'tables', 'view', 'place'
    ],
    'price': [
        'price', 'prices', 'expensive', 'cheap', 'value', 'worth',
        'cost', 'costs', 'affordable', 'overpriced', 'reasonable',
        'budget', 'money', 'pricey', 'bill', 'pay', 'paid',
        'rupees', 'rs'
    ]
}

print("Aspect Keywords Defined:")
print("="*70)
for aspect, keywords in ASPECT_KEYWORDS.items():
    print(f"  {aspect.capitalize():10}: {len(keywords)} keywords")

In [ ]:
# Extract aspects
print("\nExtracting aspects from reviews...\n")

for aspect, keywords in ASPECT_KEYWORDS.items():
    reviews_df[f'has_{aspect}'] = reviews_df['cleaned_text'].apply(
        lambda x: any(keyword in str(x).lower() for keyword in keywords)
    )

print("✓ Aspect extraction completed!\n")

# Show statistics
print("Aspect Mention Statistics:")
print("="*70)
aspect_stats = {}
for aspect in ASPECT_KEYWORDS.keys():
    count = reviews_df[f'has_{aspect}'].sum()
    pct = count / len(reviews_df) * 100
    aspect_stats[aspect] = count
    print(f"  {aspect.capitalize():10}: {count:6,} reviews ({pct:5.1f}%)")

# Count multi-aspect reviews
reviews_df['aspect_count'] = sum(reviews_df[f'has_{aspect}'] for aspect in ASPECT_KEYWORDS.keys())
multi_aspect = (reviews_df['aspect_count'] > 1).sum()
print(f"\n  Multi-aspect reviews: {multi_aspect:,} ({multi_aspect/len(reviews_df)*100:.1f}%)")

In [ ]:
# Visualize aspects
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Aspect mentions
aspect_df = pd.Series(aspect_stats).sort_values(ascending=False)
colors_aspects = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
aspect_df.plot(kind='bar', ax=axes[0], color=colors_aspects)
axes[0].set_title('Aspect Mentions in Reviews', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Aspect', fontweight='bold')
axes[0].set_ylabel('Number of Reviews', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Aspect count distribution
aspect_count_dist = reviews_df['aspect_count'].value_counts().sort_index()
aspect_count_dist.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Number of Aspects per Review', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Number of Aspects', fontweight='bold')
axes[1].set_ylabel('Number of Reviews', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Show example reviews by aspect
print("\n📝 Example Reviews by Aspect:")
print("="*70)

for aspect in ASPECT_KEYWORDS.keys():
    sample = reviews_df[reviews_df[f'has_{aspect}'] == True]['review_text'].head(2)
    
    if len(sample) > 0:
        print(f"\n{aspect.upper()}:")
        for i, review in enumerate(sample, 1):
            print(f"  {i}. {review[:120]}...")
        print("-" * 70)

## Step 9: Aspect-Sentiment Analysis

In [ ]:
# Sentiment distribution by aspect
print("\n📊 Sentiment Distribution by Aspect:")
print("="*70)

aspect_sentiment_data = []

for aspect in ASPECT_KEYWORDS.keys():
    aspect_reviews = reviews_df[reviews_df[f'has_{aspect}'] == True]
    
    if len(aspect_reviews) > 0:
        sentiment_dist = aspect_reviews['sentiment'].value_counts(normalize=True) * 100
        
        print(f"\n{aspect.capitalize()}:")
        for sentiment in ['positive', 'neutral', 'negative']:
            pct = sentiment_dist.get(sentiment, 0)
            print(f"  {sentiment:10}: {pct:5.1f}%")
        
        aspect_sentiment_data.append({
            'Aspect': aspect.capitalize(),
            'Positive': sentiment_dist.get('positive', 0),
            'Neutral': sentiment_dist.get('neutral', 0),
            'Negative': sentiment_dist.get('negative', 0)
        })

In [ ]:
# Visualize aspect-sentiment relationship
aspect_sent_df = pd.DataFrame(aspect_sentiment_data)

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(aspect_sent_df))
width = 0.25

ax.bar(x - width, aspect_sent_df['Positive'], width, label='Positive', color='#2ca02c')
ax.bar(x, aspect_sent_df['Neutral'], width, label='Neutral', color='#ffbb78')
ax.bar(x + width, aspect_sent_df['Negative'], width, label='Negative', color='#d62728')

ax.set_xlabel('Aspect', fontweight='bold', fontsize=12)
ax.set_ylabel('Percentage (%)', fontweight='bold', fontsize=12)
ax.set_title('Sentiment Distribution by Aspect', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(aspect_sent_df['Aspect'])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

print("Splitting data: 70% train, 15% validation, 15% test...\n")

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    reviews_df,
    test_size=0.3,
    random_state=42,
    stratify=reviews_df['label']
)

# Second split: 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['label']
)

print("✓ Data split completed!\n")
print("Split Summary:")
print("="*70)
print(f"  Training:   {len(train_df):6,} reviews ({len(train_df)/len(reviews_df)*100:5.1f}%)")
print(f"  Validation: {len(val_df):6,} reviews ({len(val_df)/len(reviews_df)*100:5.1f}%)")
print(f"  Test:       {len(test_df):6,} reviews ({len(test_df)/len(reviews_df)*100:5.1f}%)")
print(f"  Total:      {len(reviews_df):6,} reviews")

# Check distribution
print("\nSentiment Distribution in Splits:")
print("="*70)
for name, dataset in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n{name}:")
    for sentiment in ['positive', 'neutral', 'negative']:
        count = (dataset['sentiment'] == sentiment).sum()
        pct = count / len(dataset) * 100
        print(f"  {sentiment:10}: {count:5,} ({pct:5.1f}%)")

## Step 11: Save Preprocessed Data

In [ ]:
# Save full preprocessed dataset
output_file = 'zomato_reviews_preprocessed.csv'
reviews_df.to_csv(output_file, index=False)
print(f"✓ Full dataset saved: {output_file}")
print(f"  Rows: {len(reviews_df):,}")
print(f"  Columns: {len(reviews_df.columns)}")

# Save splits
train_df.to_csv('train_data.csv', index=False)
val_df.to_csv('val_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)

print(f"\n✓ Data splits saved:")
print(f"  - train_data.csv ({len(train_df):,} rows)")
print(f"  - val_data.csv ({len(val_df):,} rows)")
print(f"  - test_data.csv ({len(test_df):,} rows)")

## Step 12: Final Summary

In [ ]:
print("\n" + "="*70)
print("📊 PREPROCESSING SUMMARY")
print("="*70)

print(f"\n✓ Dataset: Zomato Bangalore Restaurant Reviews")
print(f"✓ Source restaurants: {len(df):,}")
print(f"✓ Total reviews extracted: {len(reviews_df):,}")
print(f"✓ Average reviews per restaurant: {len(reviews_df)/len(df):.1f}")

print(f"\n✓ Sentiment Distribution:")
for sentiment, count in reviews_df['sentiment'].value_counts().items():
    print(f"    {sentiment.capitalize()}: {count:,} ({count/len(reviews_df)*100:.1f}%)")

print(f"\n✓ Aspect Coverage:")
for aspect in ASPECT_KEYWORDS.keys():
    count = reviews_df[f'has_{aspect}'].sum()
    print(f"    {aspect.capitalize()}: {count:,} ({count/len(reviews_df)*100:.1f}%)")

print(f"\n✓ Data Splits:")
print(f"    Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

print("\n" + "="*70)
print("✅ PREPROCESSING COMPLETE!")
print("="*70)
print("\nNext: Open Part2_Train_Models.ipynb to train BERT, RoBERTa, XLM-RoBERTa!")
print("="*70)

---

## 🎉 Part 1 Complete!

### What You've Accomplished:
✅ Loaded Zomato Bangalore dataset (51K restaurants)  
✅ Extracted individual reviews from `reviews_list` column  
✅ Parsed ratings and cleaned text  
✅ Created sentiment labels (positive/neutral/negative)  
✅ Extracted 4 aspects (Food, Service, Ambiance, Price)  
✅ Analyzed aspect-sentiment relationships  
✅ Split data (70/15/15)  
✅ Saved preprocessed files  

### Files Created:
- `zomato_reviews_preprocessed.csv` - Full dataset
- `train_data.csv` - Training set
- `val_data.csv` - Validation set
- `test_data.csv` - Test set

### Dataset Stats:
- **Total reviews:** ~40K-60K (extracted from 51K restaurants)
- **Average length:** Varies
- **Aspects identified:** Food, Service, Ambiance, Price
- **Ready for training:** Yes!

---